# Arrhenius Kinetics of LLM Inference — Demo Notebook

This notebook demonstrates the **Arrhenius Kinetics of LLM Inference** protocol applied to `mistralai/ministral-8b-2512` on TIGER-Lab/MMLU-Pro.

Key additions over iter_2:
- **(A)** Model smoke-test selection loop — ministral-8b-2512 selected with 100% logprob rate and 70% OC rate
- **(B)** Explicit **two-token-dominance** PASS/FAIL verdict: `rho(Ea, Δ) > 0.6` AND `CV(log A) < 0.4` must both hold simultaneously
- **(C)** DUAL N=16 BON sampling: regression-Ea strategy vs. Delta-approx strategy for Δ ∈ {0.2, 0.3, 0.4}
- **(D)** McNemar's exact test on per-instance boolean BON outcomes

The **Arrhenius model** treats `log P(correct | T) = −Ea/T + log A` where:
- `Ea` = activation energy (difficulty of the overconfident-error instance)
- `A` = pre-exponential factor (maximum achievable accuracy at infinite T)
- `T` = sampling temperature

**This demo loads pre-computed results** from `mini_demo_data.json` (no API calls required).

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# loguru — NOT pre-installed on Colab, always install
_pip('loguru==0.7.2')

# Core packages — pre-installed on Colab, install locally only to match Colab env
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scikit-learn==1.6.1', 'scipy==1.16.3',
         'matplotlib==3.10.0', 'statsmodels==0.14.6')

In [ ]:
import json
import math
import os
import sys
from copy import deepcopy
from typing import Optional

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.linear_model import LinearRegression
from loguru import logger

# Visualization imports (added for notebook)
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

logger.remove()
logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")
print("Imports OK")

## Data Loading

Load pre-computed results from `mini_demo_data.json`. This file contains:
- `metadata`: full experiment metadata including aggregate statistics and verdicts
- `datasets[0].examples`: 15 MMLU-Pro questions (7 overconfident-error instances + 8 normal)
- `datasets[0].oc_instances`: 7 OC instances with pre-computed `p_correct_by_T` curves and Arrhenius fits

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-ce2af9-arrhenius-kinetics-of-llm-inference-acti/main/round-4/experiment-1/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f: return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
meta = data["metadata"]
dataset_entry = data["datasets"][0]
examples = dataset_entry["examples"]
oc_instances = dataset_entry["oc_instances"]

print(f"Loaded {len(examples)} examples ({sum(1 for e in examples if e.get('predict_is_oc_error') == 'true')} OC errors)")
print(f"Loaded {len(oc_instances)} OC instances with Arrhenius curves")
print(f"Experiment: {meta['experiment_id']} | Model: {meta['model_name']}")
print(f"Dataset: {meta['dataset']} | Cost: ${meta['cumulative_cost_usd']:.4f}")

## Configuration

Tunable parameters from the original script. In the full run these are the actual values used.

In [ ]:
# Temperature grid for Arrhenius sweep
TEMP_GRID = [0.05, 0.1, 0.2, 0.3, 0.5, 0.7, 1.0]

# Temperatures used for TURN entropy computation
TURN_TEMPS = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3]

# Number of samples per temperature in sweep (original=8)
N_SAMPLES = 8

# Number of samples for Best-of-N evaluation (original=16)
N_BON = 16

# Maximum OC instances to collect
MAX_OC_INSTANCES = 60

# API cost limit (USD)
COST_LIMIT = 4.0

# Valid answer letters for MMLU-Pro (10-option format)
VALID_LETTERS = "ABCDEFGHIJ"

# Bootstrap samples for R² confidence intervals
N_BOOT = 500

print("Config set. TEMP_GRID:", TEMP_GRID)

## Phase 1: Data Exploration — MMLU-Pro Examples

The dataset contains pilot, main, and catalysis splits. OC (overconfident-error) instances are those where the model answers incorrectly with high confidence at T=1.0.

In [ ]:
# Show subject and split distribution
from collections import Counter

oc_ex = [e for e in examples if e.get('predict_is_oc_error') == 'true']
non_oc_ex = [e for e in examples if e.get('predict_is_oc_error') == 'false']

print(f"OC errors: {len(oc_ex)} | Non-OC: {len(non_oc_ex)}")
print(f"\nSubject distribution:")
for subj, cnt in Counter(e['metadata_subject'] for e in examples).most_common():
    print(f"  {subj}: {cnt}")

print(f"\nSplit distribution:")
for split, cnt in Counter(e['metadata_split'] for e in examples).most_common():
    print(f"  {split}: {cnt}")

# Show an example OC error
print("\n--- Example OC error question (truncated) ---")
if oc_instances:
    inst = oc_instances[0]
    print(f"Subject: {inst['subject']} | Answer: {inst['correct_answer']}")
    print(inst['input_preview'][:300])

## Phase 4 (core): Arrhenius Fitting Functions

The Arrhenius model: `log P(correct|T) = −Ea/T + log A`

- **`fit_arrhenius`**: Linear regression on `(1/T, log P)` points; requires ≥3 valid temperature points (P > 0)
- **`fit_alternatives`**: Fits linear, exp-T, and power-law models as comparison baselines
- **`bootstrap_R2`**: 95% CI on R² via bootstrap resampling
- **`T_thresh`**: Analytical threshold `T = Ea / log(N)` below which BON@N fails

In [ ]:
def fit_arrhenius(p_correct_by_T: dict) -> Optional[dict]:
    Ts = np.array(TEMP_GRID)
    Ps = np.array([p_correct_by_T.get(T, 0.0) for T in Ts])
    mask = Ps > 0
    if mask.sum() < 3:
        return None
    inv_T = (1.0 / Ts[mask]).reshape(-1, 1)
    log_P = np.log(Ps[mask])
    reg = LinearRegression().fit(inv_T, log_P)
    log_P_pred = reg.predict(inv_T)
    ss_res = float(np.sum((log_P - log_P_pred) ** 2))
    ss_tot = float(np.sum((log_P - log_P.mean()) ** 2))
    R2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else 0.0
    slope = float(reg.coef_[0])
    Ea = -slope
    log_A = float(reg.intercept_)
    return {
        "Ea": Ea,
        "log_A": log_A,
        "R2": R2,
        "n_valid": int(mask.sum()),
        "slope": slope,
        "valid_temps": Ts[mask].tolist(),
    }


def fit_alternatives(p_correct_by_T: dict) -> dict:
    Ts = np.array(TEMP_GRID)
    Ps = np.array([p_correct_by_T.get(T, 0.0) for T in Ts])
    mask = Ps > 0
    if mask.sum() < 3:
        return {"linear": None, "exp_T": None, "power_law": None}
    Ts_v = Ts[mask]
    Ps_v = Ps[mask]

    def ols_r2(X: np.ndarray, y: np.ndarray) -> float:
        if len(np.unique(y)) < 2:
            return 0.0
        reg = LinearRegression().fit(X.reshape(-1, 1), y)
        pred = reg.predict(X.reshape(-1, 1))
        ss_res = float(np.sum((y - pred) ** 2))
        ss_tot = float(np.sum((y - y.mean()) ** 2))
        return 1.0 - ss_res / ss_tot if ss_tot > 0 else 0.0

    r2_linear = ols_r2(Ts_v, Ps_v)
    r2_exp_T = ols_r2(Ts_v, np.log(Ps_v))
    log_Tv = np.log(Ts_v)
    r2_power = ols_r2(log_Tv, np.log(Ps_v)) if len(np.unique(log_Tv)) >= 2 else None
    return {"linear": r2_linear, "exp_T": r2_exp_T, "power_law": r2_power}


def bootstrap_R2(p_correct_by_T: dict, n_boot: int = N_BOOT) -> tuple:
    Ts = np.array(TEMP_GRID)
    Ps = np.array([p_correct_by_T.get(T, 0.0) for T in Ts])
    mask = Ps > 0
    if mask.sum() < 3:
        return (None, None)
    inv_T = 1.0 / Ts[mask]
    log_P = np.log(Ps[mask])
    n = int(mask.sum())
    rng = np.random.default_rng(42)
    r2_boots = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        x_b = inv_T[idx].reshape(-1, 1)
        y_b = log_P[idx]
        if len(np.unique(y_b)) < 2:
            continue
        reg = LinearRegression().fit(x_b, y_b)
        pred = reg.predict(x_b)
        ss_res = float(np.sum((y_b - pred) ** 2))
        ss_tot = float(np.sum((y_b - y_b.mean()) ** 2))
        r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else 0.0
        r2_boots.append(r2)
    if not r2_boots:
        return (None, None)
    arr = np.array(r2_boots)
    return (float(np.percentile(arr, 2.5)), float(np.percentile(arr, 97.5)))


print("Arrhenius fitting functions defined.")

## Statistics Utilities

- **`wilson_ci`**: Wilson score interval for proportions
- **`spearman_with_ci`**: Spearman ρ with bootstrap 95% CI
- **`mcnemar_exact`**: McNemar's exact test for comparing two BON strategies

In [ ]:
def wilson_ci(k: int, n: int, z: float = 1.96) -> tuple:
    if n == 0:
        return (0.0, 0.0)
    p_hat = k / n
    denom = 1 + z ** 2 / n
    centre = (p_hat + z ** 2 / (2 * n)) / denom
    margin = z * math.sqrt(p_hat * (1 - p_hat) / n + z ** 2 / (4 * n ** 2)) / denom
    return (max(0.0, centre - margin), min(1.0, centre + margin))


def spearman_with_ci(x, y, n_boot: int = 1000, seed: int = 42) -> dict:
    x = np.array(x, dtype=float)
    y = np.array(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    x, y = x[mask], y[mask]
    if len(x) < 5:
        return {"rho": None, "p": None, "ci_low": None, "ci_high": None, "n": int(len(x))}
    rho, p_val = stats.spearmanr(x, y)
    rng = np.random.default_rng(seed)
    boots = []
    n = len(x)
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        r, _ = stats.spearmanr(x[idx], y[idx])
        if np.isfinite(r):
            boots.append(r)
    arr = np.array(boots) if boots else np.array([rho])
    return {
        "rho": float(rho),
        "p": float(p_val),
        "ci_low": float(np.percentile(arr, 2.5)),
        "ci_high": float(np.percentile(arr, 97.5)),
        "n": n,
    }


def contingency_2x2(correct_a: list, correct_b: list) -> np.ndarray:
    """Build 2×2 McNemar contingency table from two boolean lists."""
    assert len(correct_a) == len(correct_b), "Lists must have same length"
    both_T = sum(int(a and b) for a, b in zip(correct_a, correct_b))
    only_a = sum(int(a and not b) for a, b in zip(correct_a, correct_b))
    only_b = sum(int(not a and b) for a, b in zip(correct_a, correct_b))
    both_F = sum(int(not a and not b) for a, b in zip(correct_a, correct_b))
    return np.array([[both_T, only_a], [only_b, both_F]])


def mcnemar_exact(correct_a: list, correct_b: list) -> Optional[float]:
    """McNemar's exact test. Returns p-value or None if n<2."""
    if len(correct_a) < 2:
        return None
    try:
        from statsmodels.stats.contingency_tables import mcnemar as mcnemar_test
        table = contingency_2x2(correct_a, correct_b)
        result = mcnemar_test(table, exact=True, correction=False)
        return float(result.pvalue)
    except Exception as e:
        # Manual fallback: binomial test on discordant cells
        logger.warning(f"statsmodels mcnemar failed ({e}), using manual fallback")
        try:
            from scipy.stats import binomtest
            table = contingency_2x2(correct_a, correct_b)
            n_10 = int(table[0, 1])  # only_a
            n_01 = int(table[1, 0])  # only_b
            n_disc = n_10 + n_01
            if n_disc == 0:
                return 1.0
            result = binomtest(min(n_10, n_01), n=n_disc, p=0.5, alternative="two-sided")
            return float(result.pvalue)
        except Exception as e2:
            logger.error(f"McNemar fallback also failed: {e2}")
            return None


print("Statistics utilities defined.")

## Phase 4: Apply Arrhenius Fitting to OC Instances

For each overconfident-error instance, we apply the Arrhenius fit to its pre-computed `p_correct_by_T` curve.  
The fit gives us `Ea` (activation energy) and `log A` (pre-exponential), plus `R²` goodness-of-fit.

We also compute `T_thresh(N)`: the simplified analytical threshold `Ea / log(N)` below which Best-of-N@N fails.

In [ ]:
per_instance_fits = {}

for inst in oc_instances:
    qid = str(inst['question_id'])
    # p_correct_by_T keys may be strings; convert to float
    p_by_T = {float(k): v for k, v in inst['p_correct_by_T'].items()}

    arr = fit_arrhenius(p_by_T)
    alts = fit_alternatives(p_by_T)
    if arr is None:
        per_instance_fits[qid] = None
        continue

    r2_ci = bootstrap_R2(p_by_T)
    Ea = arr["Ea"]
    log_A = arr["log_A"]
    A_hat = math.exp(log_A)

    r2_vals = [v for v in alts.values() if v is not None]
    r2_advantage = arr["R2"] - max(r2_vals) if r2_vals else None

    # T_thresh per BON size N
    T_thresh_by_N: dict = {}
    for N in [4, 8, 16, 32]:
        ln_N = math.log(N)
        T_simplified = Ea / ln_N if ln_N > 0 else None
        T_thresh_by_N[N] = {"T_simplified": T_simplified}

    T_TURN = inst.get('T_TURN', 0.8)
    T_pref = max(p_by_T, key=p_by_T.get) if p_by_T else None

    per_instance_fits[qid] = {
        "qid": qid,
        "subject": inst['subject'],
        "Ea": Ea,
        "log_A": log_A,
        "A_hat": A_hat,
        "R2_arrhenius": arr["R2"],
        "R2_ci": list(r2_ci),
        "R2_alternatives": alts,
        "R2_advantage": r2_advantage,
        "T_thresh_by_N": T_thresh_by_N,
        "T_TURN": T_TURN,
        "T_pref": T_pref,
        "p_correct_by_T": p_by_T,
        "bon16_correct_regression": inst.get('bon16_correct_regression', True),
        "bon16_correct_fixed_T07": inst.get('bon16_correct_fixed_T07', True),
        "bon16_correct_delta_approx": inst.get('bon16_correct_delta_approx', True),
    }

valid_fits = [f for f in per_instance_fits.values() if f is not None]
print(f"Arrhenius fits: {len(valid_fits)}/{len(oc_instances)} valid")
print(f"\nPer-instance Ea values:")
for f in valid_fits:
    print(f"  qid={f['qid']} | subject={f['subject']} | Ea={f['Ea']:.4f} | R²={f['R2_arrhenius']:.3f}")

## Phase 5: Two-Token Dominance Test — Ea vs. Δ

The **two-token dominance** hypothesis: the Arrhenius `Ea` should correlate with `Δ = logit(wrong) − logit(correct)` at T=1.0.  
If `rho(Ea, Δ) > 0.6` AND `CV(log A) < 0.4`, the model's behavior is dominated by two-token competition.

Note: With only 7 OC instances the Spearman CI is wide — the test requires `n ≥ 5` for non-null output.

In [ ]:
def run_step5_spearman(per_instance_fits: dict) -> dict:
    valid = [f for f in per_instance_fits.values() if f is not None]
    Ea_list = [f["Ea"] for f in valid]
    log_A_list = [f["log_A"] for f in valid]

    # Use pre-computed Delta from oc_instances metadata
    # (In full run, Delta = logit_wrong - logit_correct at T=1.0)
    Delta_list = [meta['step5_Ea_vs_Delta']['rho_ea_delta']] * len(valid)  # placeholder from aggregate

    log_A_arr = np.array(log_A_list, dtype=float)
    mean_logA = float(np.mean(log_A_arr))
    std_logA = float(np.std(log_A_arr))
    cv_log_A = std_logA / abs(mean_logA) if mean_logA != 0 else None

    # Use reported rho_ea_delta from metadata (actual experiment result)
    rho_ea_delta = meta['step5_Ea_vs_Delta']['rho_ea_delta']

    # TWO-TOKEN DOMINANCE VERDICT: both thresholds must pass simultaneously
    two_token_dominance_pass = bool(
        rho_ea_delta is not None and rho_ea_delta > 0.6
        and cv_log_A is not None and cv_log_A < 0.4
    )

    return {
        "n_instances": len(valid),
        "rho_ea_delta": rho_ea_delta,
        "logA_mean": mean_logA,
        "logA_std": std_logA,
        "logA_CV": cv_log_A,
        "cv_log_A": cv_log_A,
        "two_token_dominance_pass": two_token_dominance_pass,
    }


step5 = run_step5_spearman(per_instance_fits)
print(f"Two-Token Dominance Test:")
print(f"  rho(Ea, Delta) = {step5['rho_ea_delta']:.3f}  (threshold: > 0.6)")
print(f"  CV(log A)      = {step5['cv_log_A']:.3f}  (threshold: < 0.4)")
print(f"  VERDICT: {'PASS' if step5['two_token_dominance_pass'] else 'FAIL'}")

## Phase 6: T_thresh Validation

Validates that `T_simplified = Ea / log(N)` is a conservative lower bound on the empirical minimum temperature needed for BON@N to succeed.

Also checks that `T_TURN > T_thresh@N=16` for each instance — confirming T_TURN (entropy inflection temperature) falls within the viable operating window.

In [ ]:
def run_step6_aggregate(per_instance_fits: dict) -> dict:
    valid = [f for f in per_instance_fits.values() if f is not None]
    result = {}
    for N in [4, 8, 16, 32]:
        T_simpl_vals = [f["T_thresh_by_N"][N]["T_simplified"] for f in valid]
        T_TURN_vals = [f["T_TURN"] for f in valid]
        # Check T_TURN > T_simplified (window fraction)
        n_window = sum(1 for ts, tt in zip(T_simpl_vals, T_TURN_vals)
                       if ts is not None and tt is not None and tt > ts)
        n_total = sum(1 for ts in T_simpl_vals if ts is not None)
        result[N] = {
            "n_total": n_total,
            "T_simplified_values": [round(v, 3) for v in T_simpl_vals if v is not None],
            "fraction_TURN_above_thresh": n_window / n_total if n_total else None,
        }
    pairs = [
        (f["T_thresh_by_N"][16]["T_simplified"], f["T_TURN"])
        for f in valid
        if f["T_thresh_by_N"][16]["T_simplified"] is not None and f["T_TURN"] is not None
    ]
    frac_window = sum(1 for ts, tt in pairs if tt > ts) / len(pairs) if pairs else None
    return {"by_N": result, "window_fraction_T_TURN_above_T_thresh": frac_window}


step6 = run_step6_aggregate(per_instance_fits)
print("T_thresh validation (T_simplified = Ea / log(N)):")
print(f"{'N':>5} | {'T_simplified (range)':>28} | T_TURN > T_thresh")
print("-" * 55)
for N in [4, 8, 16, 32]:
    d = step6["by_N"][N]
    vals = d["T_simplified_values"]
    frac = d["fraction_TURN_above_thresh"]
    val_range = f"{min(vals):.3f}–{max(vals):.3f}" if vals else "N/A"
    print(f"  {N:>3} | {val_range:>28} | {frac:.2f}" if frac is not None else f"  {N:>3} | {val_range:>28} | N/A")
print(f"\nWindow fraction (T_TURN > T_thresh@N=16): {step6['window_fraction_T_TURN_above_T_thresh']:.2f}")

## Phase 7: BON Comparison + McNemar Test

Two sampling strategies are evaluated against baselines using N=16 samples per instance:

| Strategy | Temperature used |
|---|---|
| Regression-Ea | `T_simplified(N=16) + Δ` |
| Delta-approx | `T_approx(N=16) + Δ` |
| Fixed T=0.7 | 0.7 (baseline) |
| Fixed T=1.0 | 1.0 (baseline) |
| TURN-adapted | Entropy-inflection T |

**McNemar's exact test** compares per-instance boolean outcomes between pairs of strategies.

In [ ]:
# Use pre-computed BON outcomes from oc_instances (stored in mini_demo_data)
correct_regression = [f["bon16_correct_regression"] for f in valid_fits]
correct_approx = [f["bon16_correct_delta_approx"] for f in valid_fits]
correct_T07 = [f["bon16_correct_fixed_T07"] for f in valid_fits]

n = len(valid_fits)
acc_regression = sum(correct_regression) / n if n else 0
acc_approx = sum(correct_approx) / n if n else 0
acc_T07 = sum(correct_T07) / n if n else 0

# McNemar tests (delta=0.3)
p_reg_vs_T07 = mcnemar_exact(correct_regression, correct_T07)
p_approx_vs_T07 = mcnemar_exact(correct_approx, correct_T07)
p_approx_vs_reg = mcnemar_exact(correct_approx, correct_regression)

print("BON@16 Accuracy Comparison (from pre-computed results):")
print(f"  Regression-Ea strategy: {acc_regression:.3f} ({int(acc_regression*n)}/{n})")
print(f"  Delta-approx strategy:  {acc_approx:.3f} ({int(acc_approx*n)}/{n})")
print(f"  Fixed T=0.7 (baseline): {acc_T07:.3f} ({int(acc_T07*n)}/{n})")
print(f"\nMcNemar's exact test:")
print(f"  Regression vs T=0.7:  p = {p_reg_vs_T07}")
print(f"  Delta-approx vs T=0.7: p = {p_approx_vs_T07}")
print(f"  Delta-approx vs Regression: p = {p_approx_vs_reg}")
print(f"\nReference (full experiment): regression={meta['bon16_accuracy_regression']:.3f}, "
      f"fixed_T07={meta['bon16_accuracy_fixed_T07']:.3f}")

## Aggregate Results & Verdict

Seven confirmation criteria (C1–C9) from the pre-registered protocol. The full experiment verdict is `CONFIRM` (4/7 criteria met).

In [ ]:
# Display aggregate results from the full run (stored in metadata)
agg = meta['aggregate']
confirm_flags = agg['confirm_flags']

print(f"Aggregate Arrhenius Statistics (n={agg['n_instances']} instances):")
r2d = agg['R2_distribution']
print(f"  R² median={r2d['median']:.3f}, mean={r2d['mean']:.3f}, p25={r2d['p25']:.3f}, p75={r2d['p75']:.3f}")
ead = agg['Ea_distribution']
print(f"  Ea  median={ead['median']:.3f}, mean={ead['mean']:.3f}, std={ead['std']:.3f}")
ttd = agg['T_TURN_distribution']
print(f"  T_TURN median={ttd['median']:.2f}, mean={ttd['mean']:.3f}")

print(f"\nConfirmation Criteria:")
criteria_desc = {
    'C1_median_R2_gt_085': 'C1: Median R² > 0.85',
    'C2_spearman_Ea_Delta_gt_05': 'C2: Spearman rho(Ea,Δ) > 0.5',
    'C3_lower_bound_N16_gt_060': 'C3: T_simplified lower-bound fraction > 0.60',
    'C4_window_fraction_ge_070': 'C4: Window fraction T_TURN > T_thresh ≥ 0.70',
    'C6_Ea_predicts_Tpref': 'C6: Ea predicts T_pref (rho > 0.3)',
    'C7_T_operating_beats_fixed': 'C7: T_operating strategy beats fixed T=0.7',
    'C9_catalysis_fraction_gt_050': 'C9: CoT reduces Ea > 50%',
}
for key, desc in criteria_desc.items():
    val = confirm_flags.get(key, False)
    print(f"  {'✓' if val else '✗'} {desc}")

print(f"\nCriteria confirmed: {agg['n_criteria_confirmed']}/7")
print(f"Overall VERDICT: {meta['verdict']}")
print(f"Two-Token Dominance: {'PASS' if meta['two_token_dominance_confirmed'] else 'FAIL'} "
      f"(rho={meta['rho_ea_delta']:.3f}, CV={meta['cv_log_A']:.3f})")

## Visualization

Four panels:
1. **Arrhenius curves** — `log P(correct)` vs `1/T` per OC instance with fitted lines
2. **R² distribution** — histogram of goodness-of-fit across instances
3. **Ea distribution** — activation energies per instance
4. **BON accuracy comparison** — regression vs. delta-approx vs. fixed-T baselines

In [ ]:
fig = plt.figure(figsize=(14, 10))
gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.4, wspace=0.35)

colors = plt.cm.tab10(np.linspace(0, 1, max(len(valid_fits), 1)))

# ── Panel 1: Arrhenius curves ──────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
T_fine = np.linspace(0.04, 1.05, 200)
for i, f in enumerate(valid_fits):
    p_by_T = f["p_correct_by_T"]
    Ts = np.array([T for T, P in sorted(p_by_T.items()) if P > 0])
    Ps = np.array([P for T, P in sorted(p_by_T.items()) if P > 0])
    if len(Ts) < 2:
        continue
    ax1.scatter(1.0 / Ts, np.log(Ps), color=colors[i], s=40, zorder=3)
    # Fitted Arrhenius line
    ax1.plot(1.0 / T_fine,
             f["log_A"] - f["Ea"] / T_fine,
             color=colors[i], alpha=0.6, linewidth=1.2,
             label=f"{f['subject'][:6]} Ea={f['Ea']:.3f}")
ax1.set_xlabel("1/T")
ax1.set_ylabel("log P(correct | T)")
ax1.set_title("Arrhenius Fits (log P vs 1/T)")
ax1.legend(fontsize=6, loc="upper right")
ax1.grid(True, alpha=0.3)

# ── Panel 2: R² distribution ──────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
r2_vals = [f["R2_arrhenius"] for f in valid_fits]
ax2.bar(range(len(r2_vals)), sorted(r2_vals, reverse=True), color=colors[:len(r2_vals)])
ax2.axhline(y=np.median(r2_vals), color='red', linestyle='--', linewidth=1.5,
            label=f'Median R²={np.median(r2_vals):.3f}')
ax2.axhline(y=0.85, color='gray', linestyle=':', linewidth=1.2, label='C1 threshold (0.85)')
ax2.set_xlabel("Instance (sorted)")
ax2.set_ylabel("R²")
ax2.set_title("R² Goodness-of-Fit (Arrhenius)")
ax2.legend(fontsize=8)
ax2.set_ylim(0, 1)
ax2.grid(True, alpha=0.3, axis='y')

# ── Panel 3: Ea distribution ──────────────────────────────────────────────────
ax3 = fig.add_subplot(gs[1, 0])
Ea_vals = [f["Ea"] for f in valid_fits]
subjects = [f["subject"] for f in valid_fits]
bars = ax3.bar(range(len(Ea_vals)), Ea_vals, color=colors[:len(Ea_vals)])
ax3.axhline(y=np.median(Ea_vals), color='red', linestyle='--', linewidth=1.5,
            label=f'Median Ea={np.median(Ea_vals):.3f}')
ax3.set_xticks(range(len(subjects)))
ax3.set_xticklabels([s[:5] for s in subjects], rotation=30, ha='right', fontsize=8)
ax3.set_ylabel("Ea (activation energy)")
ax3.set_title("Ea Per OC Instance by Subject")
ax3.legend(fontsize=8)
ax3.grid(True, alpha=0.3, axis='y')

# ── Panel 4: BON accuracy comparison ─────────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 1])
# Use full-run metadata for the strategy comparison
acc_data = meta['step7_accuracy_comparison']['accuracy']
strategies = {
    'Regr. Δ=0.2': acc_data.get('T_operating_regression_delta_0.2', 0),
    'Regr. Δ=0.3': acc_data.get('T_operating_regression_delta_0.3', 0),
    'Regr. Δ=0.4': acc_data.get('T_operating_regression_delta_0.4', 0),
    'Approx Δ=0.2': acc_data.get('T_operating_approx_delta_0.2', 0),
    'Approx Δ=0.3': acc_data.get('T_operating_approx_delta_0.3', 0),
    'Approx Δ=0.4': acc_data.get('T_operating_approx_delta_0.4', 0),
    'Fixed T=0.7': acc_data.get('fixed_T07', 0),
    'Fixed T=1.0': acc_data.get('fixed_T10', 0),
    'TURN': acc_data.get('TURN_adapted', 0),
}
strat_names = list(strategies.keys())
strat_vals = list(strategies.values())
bar_colors = ['#2196F3'] * 3 + ['#4CAF50'] * 3 + ['#FF5722', '#FF5722', '#9C27B0']
ax4.bar(range(len(strat_names)), strat_vals, color=bar_colors)
ax4.set_xticks(range(len(strat_names)))
ax4.set_xticklabels(strat_names, rotation=40, ha='right', fontsize=7)
ax4.set_ylabel("BON@16 Accuracy")
ax4.set_title("BON@16 Strategy Comparison (Full Run)")
ax4.set_ylim(0, 1.1)
ax4.grid(True, alpha=0.3, axis='y')

# Legend
from matplotlib.patches import Patch
legend_els = [Patch(color='#2196F3', label='Regression-Ea'),
              Patch(color='#4CAF50', label='Delta-approx'),
              Patch(color='#FF5722', label='Fixed-T baseline'),
              Patch(color='#9C27B0', label='TURN-adapted')]
ax4.legend(handles=legend_els, fontsize=7, loc='lower right')

fig.suptitle('Arrhenius Kinetics of LLM Inference — Demo Results\n'
             f'Model: {meta["model_name"]} | Dataset: {meta["dataset"]} | '
             f'Verdict: {meta["verdict"]}',
             fontsize=11, fontweight='bold')

plt.savefig('arrhenius_demo_results.png', dpi=120, bbox_inches='tight')
plt.show()
print("Figure saved to arrhenius_demo_results.png")

## Summary Table

In [ ]:
print("=" * 65)
print("EXPERIMENT SUMMARY")
print("=" * 65)
print(f"Model selected:          {meta['model_name']}")
print(f"Logprob OK rate:         {meta['model_selection_log'][meta['model_name']]['logprob_ok_rate']:.0%}")
print(f"Pilot OC rate:           {meta['oc_rate_pilot']:.0%}  ({meta['pilot_gate']['n_oc_pilot']}/200 pilot examples)")
print(f"Valid Arrhenius fits:    {meta['n_valid_arrhenius_fits']}/{meta['n_oc_instances']}")
print(f"Median R²:               {meta['aggregate']['R2_distribution']['median']:.3f}")
print(f"Median Ea:               {meta['aggregate']['Ea_distribution']['median']:.3f}")
print(f"rho(Ea, Delta):          {meta['rho_ea_delta']:.3f}  (C2 threshold: > 0.6)")
print(f"CV(log A):               {meta['cv_log_A']:.3f}  (threshold: < 0.4)")
print(f"Two-token dominance:     {'PASS' if meta['two_token_dominance_confirmed'] else 'FAIL'}")
print(f"BON@16 regression (d=0.3): {meta['step7_accuracy_comparison']['accuracy'].get('T_operating_regression_delta_0.3', 'N/A'):.3f}")
print(f"BON@16 fixed T=0.7:      {meta['bon16_accuracy_fixed_T07']:.3f}")
print(f"McNemar p (reg vs T=0.7): {meta['step7_accuracy_comparison']['mcnemar']['delta_0.3']['mcnemar_p_regression_vs_T07']}")
print(f"Spearman rho(Ea, T_pref): {meta['step8_Ea_predicts_Tpref']['spearman_Ea_Tpref']['rho']:.3f}")
print(f"Total API calls:         {meta['total_api_calls']}")
print(f"Total cost:              ${meta['cumulative_cost_usd']:.4f}")
print(f"Overall verdict:         {meta['verdict']}")
print("=" * 65)